<a href="https://colab.research.google.com/github/sig-gis/cwcb-landcover-mapping/blob/main/00_Intial_Explorations/OpenEarthMap_dl_util.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Get the entire obtainable OpenEarthMap into Drive.
# Local disk does the work; one visible bulk copy to Drive at the end.
import os, subprocess, hashlib, glob

DRIVE_DEST = "/content/drive/MyDrive/datasets/OpenEarthMap"   # final home in Drive
XBD_SRC    = "/content/drive/MyDrive/datasets/xbd_archives"   # optional: xview2 .tar.gz files
WORK, URL  = "/content/oem", "https://zenodo.org/records/7223446/files/OpenEarthMap.zip?download=1"
MD5        = "64155d1dc9d3b68536063f79878e1a67"

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)

# 1. download to LOCAL disk + verify (-c resumes if it ever drops)
zp = f"{WORK}/OpenEarthMap.zip"
subprocess.run(["wget", "-c", "-O", zp, URL], check=True)
h = hashlib.md5()
with open(zp, "rb") as f:
    for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
assert h.hexdigest() == MD5, "MD5 mismatch — just re-run this cell, the download was truncated"
print("zip verified")

# 2. extract locally
subprocess.run(["unzip", "-q", "-o", zp, "-d", WORK], check=True)
OEM_DIR = os.path.dirname(glob.glob(f"{WORK}/**/xbd_files.csv", recursive=True)[0])

# 3. OPTIONAL: merge xBD region images IF you've put xview2 .tar.gz files in XBD_SRC.
#    No login = no xBD images; the script can't fetch them for you. Skips cleanly if absent.
arc = sorted(glob.glob(f"{XBD_SRC}/*.tar*")) if os.path.isdir(XBD_SRC) else []
if arc:
    xbd = f"{WORK}/xBD"; os.makedirs(xbd, exist_ok=True)
    for a in arc: subprocess.run(["tar", "-xf", a, "-C", xbd], check=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/bao18/open_earth_map", f"{WORK}/repo"], check=False)
    subprocess.run(["python", f"{WORK}/repo/data/compile_xbd.py",
                    "--path_to_OpenEarthMap", OEM_DIR, "--path_to_xBD", xbd], check=True)
    print("xBD images merged")
else:
    print("No xBD archives found — skipped. Disaster-region images stay blank until you "
          "download them from https://xview2.org/download (login) and re-run.")

# 4. one bulk copy to Drive, with progress so it doesn't look hung (this part takes a while)
os.makedirs(DRIVE_DEST, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2", f"{OEM_DIR}/", f"{DRIVE_DEST}/"], check=True)
print("done →", DRIVE_DEST)

Mounted at /content/drive
zip verified
No xBD archives found — skipped. Disaster-region images stay blank until you download them from https://xview2.org/download (login) and re-run.
done → /content/drive/MyDrive/datasets/OpenEarthMap
